# Unix-terminal. Работа с сетью


## Мотивация

Вы развернули что-то на сервере: запустили обучение с TensorBoard, подняли
Jupyter, отдали коллеге ссылку на свой сервис. Открываете в браузере —
«страница недоступна».

Дальше обычно начинается угадывание: перезапустить, поменять порт, написать
администратору, попробовать с телефона. Уходит час, причина так и не найдена,
а на следующей неделе всё повторяется.

Сеть устроена слоями, и **каждый слой проверяется отдельной командой за
секунды**. Если идти снизу вверх, ответ находится с третьей-четвёртой попытки,
а не с тридцатой — причём находится точное место поломки, а не «что-то с сетью».

Лестница диагностики

Сегодня мы пройдём эту лестницу целиком. К концу занятия на вопрос «почему не
работает» у вас будет ответ вида «сломано на слое 4: порт слушается только на
петле, вот вывод `ss`» — а с таким ответом чинится уже быстро.


### Рабочий каталог

Всё делаем на своей виртуальной машине в `~/seminar-07/`. Каталог `/tmp` не
берём: он вычищается при перезагрузке, а файлы нужны и после занятия.


In [ ]:
%%bash
mkdir -p ~/seminar-07
cd ~/seminar-07 || exit 1
pwd


## 1. Свой узел: адрес, интерфейс, маршрут

Нижняя ступень лестницы. Прежде чем спрашивать «почему не отвечает тот сервер»,
надо убедиться, что у нас самих есть адрес и понятно, куда пойдут пакеты.


In [ ]:
%%bash
ip -brief address


Три вещи в выводе: имя интерфейса, состояние (`UP`/`DOWN`) и адреса.
Интерфейс `lo` с адресом `127.0.0.1` — петля: она есть всегда и никуда наружу
не ведёт. Внешний интерфейс обычно называется `eth0`, `ens3` или похоже.

Адресов у машины несколько, и это нормально. Какой из них «настоящий» —
зависит от того, куда мы собрались идти. Это и показывает `ip route get`.


In [ ]:
%%bash
ip route get 8.8.8.8


Читается так: чтобы дойти до `8.8.8.8`, пакет пойдёт через шлюз `via ...`,
интерфейсом `dev ...`, и уйдёт **с адреса** `src ...`. Вот этот `src` и есть
тот адрес, который увидит собеседник.

#### ❓ **Вопрос**: У машины есть и `127.0.0.1`, и адрес вида `10.0.0.5`. На какой из них придёт запрос от коллеги с соседней машины?

<details>

<summary><strong>Ответ</strong></summary>

Только на `10.0.0.5` — тот, что показан в `src` у `ip route get`. Адрес
`127.0.0.1` есть у каждой машины свой собственный, он не выходит за пределы
хоста: коллега, обратившись к `127.0.0.1`, попадёт на самого себя.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

В старых инструкциях из интернета вы встретите `ifconfig`, `route` и `netstat`
— это пакет `net-tools`, он не развивается с середины 2000-х и в свежих
образах часто просто не установлен. Современная замена — `ip` (адреса,
маршруты, интерфейсы) и `ss` (сокеты) из пакета `iproute2`. Команды `ifconfig`
и `netstat` продолжают работать там, где их доставили, но новые ключи и типы
адресов они уже не знают, поэтому в курсе используем `ip` и `ss`.

</details>


## 2. Внешний адрес: почему их два

Адрес на интерфейсе и адрес, под которым машину видит интернет, — часто разные.
Между ними стоит NAT: домашний роутер или облачный шлюз.


In [ ]:
%%bash
ip -brief address show scope global
echo "--- а так меня видит интернет ---"
curl -4 -s -m 5 https://ifconfig.me
echo


#### ❓ **Вопрос**: `ip address` показывает `10.x.x.x`, а сервис в интернете отвечает совсем другим адресом. Кто из них врёт?

<details>

<summary><strong>Ответ</strong></summary>

Никто. `10.x.x.x` — частный адрес внутри локальной сети, он есть у миллионов
машин одновременно и в интернете не маршрутизируется. На выходе роутер
подменяет адрес отправителя на свой публичный — это и есть NAT. Практическое
следствие: снаружи к вашему `10.x.x.x` подключиться нельзя, даже если сервис
слушает на всех интерфейсах, — нужен проброс порта на роутере или туннель.

</details>


## 3. Имя: во что оно превращается

Вторая ступень. Пока имя не превратилось в адрес, соединяться не с чем.

Путь от имени к адресу


In [ ]:
%%bash
getent hosts example.com
dig +short example.com


`getent hosts` спрашивает так же, как это делают обычные программы — через
системную функцию `getaddrinfo`. `dig` — инструмент отладки DNS: он ходит в
DNS-сервер напрямую и показывает сырой ответ.

Если у машины есть IPv6, `getent hosts` вполне может вернуть адрес вида
`2606:4700::...`, а `dig +short` — привычный IPv4. Это не противоречие:
`getaddrinfo` отдаёт программе **список** адресов обоих семейств, а `dig` без
уточнений спрашивает только записи типа A, то есть IPv4.

Полный вывод `dig` полезен двумя строчками: сама A-запись с временем жизни
(TTL) и `SERVER:` — кто именно ответил.


In [ ]:
%%bash
dig example.com | sed -n '/ANSWER SECTION/,+2p'
dig example.com | grep 'SERVER:'


А теперь фокус, который объясняет половину загадочных случаев «у меня
резолвится, а программа не идёт»:


In [ ]:
%%bash
grep -w localhost /etc/hosts | head -2
echo "--- getent: так адрес получают программы ---"
getent hosts localhost
echo "--- dig у публичного DNS-сервера ---"
dig +short localhost @1.1.1.1
echo "(пусто — в публичном DNS имени localhost нет)"


#### ❓ **Вопрос**: `ping localhost` работает прекрасно, хотя в публичном DNS такого имени нет. Откуда программа берёт адрес?

<details>

<summary><strong>Ответ</strong></summary>

Из файла `/etc/hosts`. `ping` идёт через `getaddrinfo`, а тот сначала читает
этот файл и, найдя там `localhost`, дальше уже никуда не ходит. DNS в этой
истории вообще не участвует.

Отсюда рабочее правило: если `dig` показывает правильный адрес, а программа
идёт не туда — проверьте `/etc/hosts`, там наверняка лежит перекрывающая
запись. Сверять надо `getent hosts`, а не `dig`.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Тонкость, на которой легко обжечься: `dig +short localhost` **без** указания
сервера на современной Ubuntu отвечает `127.0.0.1`. Кажется, что это опровергает
всё сказанное, но нет: запрос уходит в локальный stub-резолвер
`systemd-resolved` (адрес `127.0.0.53`), а тот синтезирует ответ для `localhost`
сам, не спрашивая интернет. Именно поэтому в демонстрации мы явно указали
`@1.1.1.1` — публичный сервер.

Практический вывод шире примера: у `dig` всегда стоит спрашивать себя, **какой
резолвер** ответил. Строка `SERVER:` в полном выводе — не украшение: ответы
локального кэша, корпоративного DNS и публичного сервера могут различаться, и
половина «мистических» расхождений живёт именно здесь.

</details>


## 4. Хост: доходят ли пакеты

Третья ступень. Адрес есть — проверяем, отвечает ли машина.


In [ ]:
%%bash
ping -c 3 -W 2 example.com


Смотрим на две вещи: процент потерь и время отклика. Потери — признак плохого
канала, большое время — признак дальнего или перегруженного маршрута.

Теперь постучимся по адресу из диапазона, который зарезервирован под примеры в
документации и в интернете не маршрутизируется:


In [ ]:
%%bash
ping -c 2 -W 2 192.0.2.1 || echo "ответа нет"


#### ❓ **Вопрос**: Сервер не отвечает на `ping`. Можно ли из этого сделать вывод, что он выключен?

<details>

<summary><strong>Ответ</strong></summary>

Нельзя. `ping` использует протокол ICMP, а его очень часто режут на фаерволах
и у облачных провайдеров — специально, чтобы машину не сканировали. Живой
сервер с работающим сайтом может молчать на `ping`.

Обратное утверждение сильнее: если `ping` **отвечает** — хост точно жив и
маршрут до него есть. Поэтому успешный `ping` — хорошая новость, а неуспешный
ничего не доказывает, и надо идти на ступень выше и стучаться прямо в порт.

</details>


## 5. Порт: кто слушает у меня

Четвёртая ступень. Поднимем свой сервис, чтобы было что диагностировать.


In [ ]:
%%bash
cd ~/seminar-07 || exit 1
echo "привет из семинара 7" > index.html
nohup python3 -m http.server 8000 --bind 0.0.0.0 > server.log 2>&1 &
sleep 1
ss -tlnp | grep 8000


Ключи `ss` разбираются по буквам: `-t` — TCP, `-l` — только слушающие сокеты,
`-n` — не превращать номера портов в имена, `-p` — показать процесс-владелец.

Главная колонка — `Local Address:Port`. Слева от двоеточия стоит **адрес
привязки**, и именно он решает, кто сможет подключиться.

#### ❓ **Вопрос**: В колонке адреса написано `0.0.0.0:8000`. Что означает `0.0.0.0` — это какой-то конкретный узел?

<details>

<summary><strong>Ответ</strong></summary>

Нет, это не адрес узла, а способ сказать «любой адрес этой машины». Сервис
принимает соединения на всех интерфейсах сразу: и на петле `127.0.0.1`, и на
внешнем `10.x.x.x`. Противоположность — привязка к одному конкретному адресу,
и вот тогда начинаются сюрпризы (следующий раздел).

</details>


## 6. Ловушка: адрес привязки

Самая частая причина «на сервере работает, а из браузера нет». Порт открыт,
процесс жив, в логах пусто — а снаружи отказ.

Адрес привязки: 127.0.0.1 против 0.0.0.0

Перезапустим тот же сервер, поменяв только адрес привязки:


In [ ]:
%%bash
pkill -f "http.server 8000"
sleep 1
cd ~/seminar-07 || exit 1
nohup python3 -m http.server 8000 --bind 127.0.0.1 > server.log 2>&1 &
sleep 1
ss -tlnp | grep 8000


Порт по-прежнему слушается, процесс тот же. Проверим его с двух сторон: через
петлю и через собственный внешний адрес машины.


In [ ]:
%%bash
MY_IP=$(ip route get 8.8.8.8 | grep -oP 'src \K\S+')
echo "внешний адрес машины: $MY_IP"
curl -s -m 3 http://127.0.0.1:8000/ || echo "через петлю: не ответил"
curl -s -m 3 "http://$MY_IP:8000/" || echo "через внешний адрес: не ответил"


#### ❓ **Вопрос**: Коллега говорит «твой сервис не открывается», вы заходите на сервер, делаете `curl localhost:8000` — всё работает. Что проверить первым делом?

<details>

<summary><strong>Ответ</strong></summary>

Адрес привязки в выводе `ss -tlnp`: стоит ли там `127.0.0.1:8000` вместо
`0.0.0.0:8000`. Проверка `curl localhost` в этой ситуации бесполезна — она
проходит через петлю и будет успешной в обоих случаях. Проверять надо по тому
же адресу, по которому идёт коллега.

Многие сервисы (Jupyter, TensorBoard, dev-серверы фреймворков) привязываются к
петле **по умолчанию** — это защита, а не ошибка: так сервис не оказывается
случайно открыт всей сети.

</details>


## 7. Три исхода стука в порт

Если запомнить из семинара одну вещь — пусть это будет она.

Три исхода TCP-подключения


In [ ]:
%%bash
nc -z -v -w 3 127.0.0.1 8000 2>&1 | tail -1
nc -z -v -w 3 127.0.0.1 9 2>&1 | tail -1
nc -z -v -w 3 192.0.2.1 80 2>&1 | tail -1


Ключи: `-z` — только проверить, ничего не передавать, `-v` — рассказывать, что
происходит, `-w 3` — не ждать дольше трёх секунд.

Обратите внимание не только на текст, но и **на время**: первые две строки
появились мгновенно, третья — через три секунды.

#### ❓ **Вопрос**: В одном случае вы получили `Connection refused` сразу, в другом — «завис и отвалился по таймауту». Где чинить в каждом случае?

<details>

<summary><strong>Ответ</strong></summary>

`Connection refused` — до хоста мы дошли, он ответил пакетом RST: «здесь никто
не слушает». Сеть и маршрут в порядке, чинить надо **на самом хосте**: сервис
не запущен, упал или слушает другой порт/адрес.

Таймаут — ответа не было вообще. Пакет где-то отбросили молча: фаервол с
правилом DROP, группа безопасности в облаке, неверный адрес, выключенная
машина. Чинить надо **по дороге**, а не в сервисе.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

`nc` существует в нескольких несовместимых реализациях (openbsd, traditional,
ncat из состава nmap), и ключи у них расходятся: где-то нет `-z`, где-то
по-другому пишется таймаут. Проверять надо `nc -h`. Если `nc` вообще нет,
ту же проверку делает `curl -v telnet://host:port` или на голом bash:
`timeout 3 bash -c '</dev/tcp/127.0.0.1/8000'` — код возврата скажет то же
самое.

</details>


## 8. Приложение: разговор с сервисом

Верхняя ступень. Соединение устанавливается — теперь смотрим, что отвечает сам
сервис. Здесь работает `curl`.


In [ ]:
%%bash
curl -sS -I -m 5 https://example.com | head -4


`-I` запрашивает только заголовки, без тела. Первая строка — код ответа, и он
сразу говорит, чья это проблема.


In [ ]:
%%bash
curl -s -o /dev/null -m 5 -w 'существующая страница: %{http_code}\n' https://example.com/
curl -s -o /dev/null -m 5 -w 'выдуманный адрес:      %{http_code}\n' https://example.com/net-2026


#### ❓ **Вопрос**: Сервис вернул `404`. На какой ступени лестницы проблема?

<details>

<summary><strong>Ответ</strong></summary>

Ни на какой из нижних — все они пройдены успешно. Имя разрешилось, хост
достижим, порт слушается, сервис принял запрос, понял его и осмысленно
ответил «такого пути у меня нет». Это проблема на уровне приложения: неверный
URL или не задеплоенный обработчик.

Ровно поэтому `404` — хорошая новость при диагностике: он доказывает, что вся
сетевая часть работает.

</details>


Ключ `-w` умеет показывать не только код, но и тайминги по этапам. Это
позволяет ответить на вопрос «а где, собственно, тормозит»:


In [ ]:
%%bash
curl -s -o /dev/null -m 10 https://example.com/ \
  -w 'имя в адрес: %{time_namelookup}\nсоединение:  %{time_connect}\nTLS:         %{time_appconnect}\nпервый байт: %{time_starttransfer}\nвсего:       %{time_total}\n'


Читается по разностям: большой `time_namelookup` — тормозит DNS; разрыв между
`time_connect` и `time_starttransfer` — думает само приложение.

А `-v` показывает сам разговор: куда пошли, какой адрес выбрали, что отправили
и что получили в ответ.


In [ ]:
%%bash
curl -v -s -o /dev/null -m 5 http://127.0.0.1:8000/ 2>&1 | head -12


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

`curl` и `wget` часто путают. Грубое разделение: `curl` — «разговор с
сервисом», по умолчанию печатает ответ в stdout, умеет любые методы и
заголовки, удобен в конвейере; `wget` — «скачать файл», умеет рекурсивный
обход и докачку, по умолчанию пишет на диск. Для диагностики берут `curl`,
для «утащить архив с датасетом» — `wget`.

Работу с REST API и разбор JSON мы будем разбирать отдельно в семинаре 11;
сегодня `curl` нужен нам только как измерительный прибор.

</details>


## 9. Кто мешает: фаервол

Мы видели таймаут на неотвечающем адресе. Точно такой же таймаут получится,
если пакет отбросит фаервол на пути. Посмотреть правила на своей машине можно
так — команда требует `sudo`, поэтому её показывает преподаватель:


In [ ]:
%%bash
sudo ufw status verbose


Важная деталь про то, как фаервол отказывает. Есть два разных действия:

- **DROP** — пакет молча выбрасывается, отправителю не отвечают ничего.
  Клиент видит **таймаут**;
- **REJECT** — в ответ отправляется явный отказ. Клиент видит
  **connection refused**, как если бы сервис просто не был запущен.

По умолчанию `ufw deny` работает как DROP.

#### ❓ **Вопрос**: Почему администраторы обычно выбирают DROP, а не REJECT, хотя REJECT честнее и удобнее для отладки?

<details>

<summary><strong>Ответ</strong></summary>

Молчание замедляет сканирование: чтобы перебрать порты, атакующему придётся
ждать таймаут на каждом, вместо мгновенного ответа. Плюс на отказы тратится
исходящий трафик.

Обратная сторона — ровно та боль, ради которой мы сегодня собрались:
диагностика становится неоднозначной, потому что «фаервол» и «сервис не
запущен» снаружи выглядят по-разному, а «фаервол» и «машины нет» — одинаково.

</details>


## 10. Лестница целиком

Итоговая таблица — с ней и надо подходить к недоступному сервису:

| Что видим | Ступень | Чем проверяем | Куда смотреть дальше |
|---|---|---|---|
| `Name or service not known` | 2. имя | `getent hosts`, `dig` | опечатка в имени, `/etc/hosts`, `/etc/resolv.conf` |
| `dig` отвечает, программа идёт не туда | 2. имя | `getent hosts` | перекрывающая запись в `/etc/hosts` |
| `ping` молчит, но сервис работает | 3. хост | `nc -zv` | нормально: ICMP режут, идём выше |
| `Connection refused` | 4. порт | `ss -tlnp` на сервере | сервис не запущен или слушает другой порт |
| Локально работает, снаружи `refused` | 4. порт | `ss -tlnp`, колонка адреса | привязка к `127.0.0.1` вместо `0.0.0.0` |
| Таймаут | 4. порт | `ufw status`, облачные правила | фаервол, группа безопасности, NAT, не тот адрес |
| `404`, `500`, `502` | 5. приложение | `curl -I`, логи сервиса | сеть в порядке, проблема в самом сервисе |
| Всё открывается, но медленно | 5. приложение | `curl -w` тайминги | смотрим, какой этап съел время |

Правило одно: **первая ступень, которая ответила не так, как ожидалось, и есть
место поломки**. Подниматься выше неё бессмысленно.


### Убираем за собой

Учебный сервер нам больше не нужен — снимаем его, чтобы порт не остался занят.


In [ ]:
%%bash
pkill -f "http.server 8000"
sleep 1
ss -tlnp | grep 8000 || echo "порт 8000 свободен"


## Что осталось за кадром

- **Туннели, `scp`, `rsync`, ключи** — семинар 8 про SSH. Там же приём
  «пробросить порт с сервера к себе», когда сервис намеренно слушает только
  петлю.
- **Проброс портов контейнера, сети docker** — семинар 9.
- **REST API, разбор JSON, `requests`** — семинар 11. Сегодня `curl` был
  измерительным прибором, а не клиентом API.
- **`traceroute` и `mtr`** — когда таймаут случается не у вас и не на сервере,
  а где-то посередине маршрута.
- **`tcpdump` и Wireshark** — когда нужно посмотреть на сами пакеты. Это
  следующий уровень, но начинается он всё равно с той же лестницы.
